# 原始 GraphRAG 手写教学版 · HotpotQA × Local / Global × Llama 3.3 70B

**本 Notebook 只用 HotpotQA。** 它沿用你之前的“手写完整流程”，但把每一个算法步骤拆开，并显示真正的中间产物，而不是把 GraphRAG 藏在一条 `index` 命令里。

**模型分工（沿用我们最近确定的混合方案）**：

| 流程 | 执行者 | 本 Notebook 的教学产物 |
|---|---|---|
| 数据整理、去重 | Python | 问题、候选段落、分布 |
| 实体/关系抽取 | GPT-4o-mini | 抽取 JSON、实体表、关系列表 |
| 图谱合并 | Python + NetworkX | 节点、边、证据来源 |
| 社区检测 | Leiden（igraph/leidenalg） | **彩色知识图谱 + 社区规模图** |
| 社区报告 | GPT-4o-mini | **报告正文、关键发现、对应实体和文档** |
| Embedding | OpenAI `text-embedding-3-small` | 实体/文本/报告的相似度表 |
| Local Search | 图扩展 + 向量排序 | 种子实体、相邻节点、召回文档 |
| Global Search | 社区报告 + GPT-4o-mini Map | Map 输出及其证据 |
| **最终回答** | **Colab A100 上的本地 Llama 3.3 70B** | 两种方法的结果及 EM / F1 |

> **方法边界：** 这是对 GraphRAG 核心思想的**透明、单层 Leiden 教学实现**，不是微软 2024 原论文中完整的 *hierarchical Leiden* + 官方查询引擎；Global 只对相似度最高的少数报告做 Map。不能把它称为官方精确复现。特别地，Global 的中间 Map 使用 GPT 而最终 Reduce 使用 Llama，是混合模型实验。

**使用顺序：** A100 GPU；默认 **3 题**（约 30 篇候选段落），完整观察后，把配置中的 `N_QUESTIONS=3` 改成 `200`。OpenAI 会产生 API 费用；全部步骤均支持持久化缓存。**不要将 HotpotQA gold answer/supporting facts 写入建图文本。**

## 第 0 步｜安装轻量索引/可视化依赖

这里**不安装官方 `graphrag`**，不强制升级 NumPy，也不创建 Python 3.12 子环境。大模型 `transformers` 依赖留到最后加载 Llama 时再处理。若 Colab 提示重新启动会话，请重启后从本步之后继续执行。

In [20]:
import sys, subprocess
packages = [
    "openai>=1.40,<3", "datasets>=3,<5", "pandas>=2", "tqdm>=4.66",
    "networkx>=3", "python-igraph>=0.11.9,<2", "leidenalg>=0.10.2,<1",
    "plotly>=5.20,<7", "huggingface_hub>=0.30,<2"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
print("依赖安装完成。")

依赖安装完成。


In [21]:
!pip install -q --upgrade --force-reinstall --no-deps "huggingface_hub==0.36.2"
!pip install -q --upgrade "transformers==4.57.6"

## 第 1 步｜设置实验规模、模型和缓存路径

- `N_QUESTIONS=3`：先学习流程。把它改为 200 就是正式 200 题实验。
- `OPENAI_API_KEY`：只能是 OpenAI 的密钥；`HF_TOKEN` 只用于最后下载 Llama。
- 每一步都有落盘缓存，同一设置重复运行只会处理没有完成的内容。
- 输入文档、建图、社区报告、Embedding、Global Map、最终答案都各自保留，方便检查。

In [22]:
import os, json, re, time, hashlib, string, random, warnings
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Markdown
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
import igraph as ig
import leidenalg
from openai import OpenAI

# ----------- 只需修改这里 -----------
N_QUESTIONS = 20              # 学习 3；正式实验改为 200
SEED = 42
EXTRACT_MODEL = "gpt-4o-mini"
REPORT_MODEL = "gpt-4o-mini"
MAP_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"
FINAL_LLM = "unsloth/Llama-3.3-70B-Instruct-bnb-4bit"

EXTRACT_WORKERS = 4
REPORT_WORKERS = 4
MAP_WORKERS = 4
MAX_EXTRACT_OUTPUT = 1400
MAX_REPORT_OUTPUT = 700
MAX_MAP_OUTPUT = 350
MAX_LLAMA_OUTPUT = 100
MAX_LLAMA_INPUT = 6000
LOCAL_TOP_ENTITIES = 6
LOCAL_MAX_DOCS = 7
GLOBAL_TOP_COMMUNITIES = 8
LEIDEN_RESOLUTION = 1.0
VIS_MAX_NODES = 90

random.seed(SEED); np.random.seed(SEED)
try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/GraphRAG_HotpotQA_Tutorial")
    try: key = userdata.get("OPENAI_API_KEY")
    except Exception: key = None
except ImportError:
    BASE = Path("/mnt/data/GraphRAG_HotpotQA_Tutorial")
    key = None

key = (key or os.getenv("OPENAI_API_KEY") or "").strip()
if not key:
    key = getpass("请输入 OpenAI API Key（用于 GPT-4o-mini 和 Embedding）: ").strip()
if key.startswith("hf_") or not key:
    raise ValueError("这里必须填写 OpenAI API Key，不能填写 hf_ 开头的 Hugging Face Token")
client = OpenAI(api_key=key, timeout=120, max_retries=2)
# 不将密钥保存到 Notebook/Drive；避免长期暴露在环境变量中

PUBLIC_CONFIG = dict(n=N_QUESTIONS, seed=SEED, extract=EXTRACT_MODEL,
    report=REPORT_MODEL, map=MAP_MODEL, embed=EMBED_MODEL,
    extract_max=MAX_EXTRACT_OUTPUT, report_max=MAX_REPORT_OUTPUT,
    map_max=MAX_MAP_OUTPUT, resolution=LEIDEN_RESOLUTION,
    local_entities=LOCAL_TOP_ENTITIES, local_docs=LOCAL_MAX_DOCS,
    global_communities=GLOBAL_TOP_COMMUNITIES, tutorial_revision="2026-10-11-b")
RUN_HASH = hashlib.sha256(json.dumps(PUBLIC_CONFIG, sort_keys=True).encode()).hexdigest()[:12]
ROOT = BASE / f"hotpot_n{N_QUESTIONS}_seed{SEED}_{RUN_HASH}"
ROOT.mkdir(parents=True, exist_ok=True)
for sub in ("extractions", "reports", "embeddings", "global_maps", "predictions", "figures"):
    (ROOT / sub).mkdir(exist_ok=True)
(ROOT / "config.json").write_text(json.dumps(PUBLIC_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")

def atomic_json(path, obj):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(temp, path)

def sha(value):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()[:20]

def usage_dict(usage):
    return dict(prompt_tokens=int(getattr(usage, "prompt_tokens", 0) or 0),
                completion_tokens=int(getattr(usage, "completion_tokens", 0) or 0),
                total_tokens=int(getattr(usage, "total_tokens", 0) or 0))

print("实验目录:", ROOT)
print("数据集：HotpotQA；题目：", N_QUESTIONS)
print("GPT 前处理：", EXTRACT_MODEL, "/ Embedding：", EMBED_MODEL)
print("最终回答：", FINAL_LLM)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
请输入 OpenAI API Key（用于 GPT-4o-mini 和 Embedding）: ··········
实验目录: /content/drive/MyDrive/GraphRAG_HotpotQA_Tutorial/hotpot_n20_seed42_9c3a52009c3d
数据集：HotpotQA；题目： 20
GPT 前处理： gpt-4o-mini / Embedding： text-embedding-3-small
最终回答： unsloth/Llama-3.3-70B-Instruct-bnb-4bit


## 第 2 步｜下载 HotpotQA 验证集，观察问题和支持证据

使用 `hotpotqa/hotpot_qa` 的 **distractor / validation** 划分，`shuffle(seed=42)` 后取前 N 个问题。展示 Gold Answer 仅用于理解和最后评估，**下一步写入建图库的只有候选段落本身**。

In [23]:
from datasets import load_dataset

dataset = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
questions = [dict(row) for row in dataset.shuffle(seed=SEED).select(range(N_QUESTIONS))]
for q in questions:
    q["id"] = str(q["id"])
assert len({q["id"] for q in questions}) == len(questions)

qrows = []
for q in questions:
    qrows.append(dict(question_id=q["id"], question=q["question"],
        gold_answer=q["answer"], type=q.get("type"), level=q.get("level"),
        num_candidate_paragraphs=len(q["context"]["title"]),
        num_supporting_sentences=len(q["supporting_facts"]["title"])))
qtable = pd.DataFrame(qrows)
display(qtable)
(ROOT / "sample_ids.json").write_text(json.dumps([q["id"] for q in questions], indent=2), encoding="utf-8")

sample = questions[0]
print("第一个问题:", sample["question"])
print("标准答案（只用于评估）:", sample["answer"])
print("首篇候选文档标题:", sample["context"]["title"][0])
print("段落预览:", " ".join(sample["context"]["sentences"][0])[:650])

,question_id,question,gold_answer,type,level,num_candidate_paragraphs,num_supporting_sentences
0,5add1d575542992c1e3a2540,What nationality was Oliver Reed's character i...,Prussian,bridge,hard,10,3
1,5ac55ea55542993e66e82377,Pacific Mozart Ensemble performed which German...,Kurt Julian Weill,bridge,hard,10,2
2,5ac3a76e554299741d48a2be,"Who released the song ""With or Without You"" fi...",U2,bridge,hard,10,5
3,5a7133565542994082a3e65c,"What Kentucky county has a population of 60,31...",Oldham County,bridge,hard,10,5
4,5ac1a4745542991316484b82,"Para Hills West, South Australia lies within a...","138,535",bridge,hard,10,2
5,5adc597d55429947ff17392f,"In what year was the narrator of ""Blackadder's...",1959,bridge,hard,10,2
6,5ac55e835542993e66e82375,Thomas Fitch defended John Henry Holliday when...,"October 25, 1881",bridge,hard,10,2
7,5a8d9e0f554299441c6ba015,The actor that played Gutthi on Comedy nights ...,Vaisakhi List,bridge,hard,10,2
8,5ae70ed7554299572ea546a1,"Victor Varnada directed Twisted Fortune, which...",Charlie Murphy,bridge,hard,10,3
9,5a74a8fa55429929fddd8497,After his curacy at the village that is a subu...,Dudman,bridge,hard,10,3


第一个问题: What nationality was Oliver Reed's character in the film Royal Flash?
标准答案（只用于评估）: Prussian
首篇候选文档标题: Robin Barton
段落预览: Robin Barton (born 5 November 1958) is a British art dealer dealing primarily with Banksy's.  Barton studied photography and graphic design at the Exeter College of Art and Design and this was his first encounter with Russell Young.  Moving to London in 1980 he began working as a freelance photographer for music and fashion publications "Sounds", "NME", "Blitz", "The Face" moving on to working regularly for pioneering "Independent Magazine" photographing amongst others Sir Alec Guinness, Oliver Reed, Johnny Depp, Lou Reed, Hugh Grant and Sir Peter Hall.  Laterly he worked for other publications "Sunday Times", "Sunday Telegraph", "Elle", "Vog


## 第 3 步｜把问题候选段落整理成文档集合

同一篇文本（**标题 + 全文完全相同**）只保留一次；保留 `doc_id -> 原始文本` 和 `question_id -> doc_ids` 映射。不能仅按标题去重，因为同标题的不同段落可能包含不同信息。这里同时存储 gold 支持文档 ID **仅用于评估，不供构图、检索使用**。

In [24]:
docs = {}
question_docs = {}
gold_support_docs = {}
for q in questions:
    candidates, supports = [], []
    support_titles = set(map(str, q["supporting_facts"]["title"]))
    for title, sentence_list in zip(q["context"]["title"], q["context"]["sentences"]):
        content = " ".join(str(s) for s in sentence_list).strip()
        if not content: continue
        title = str(title)
        did = sha(title + "\n" + content)
        docs.setdefault(did, {"id": did, "title": title, "text": content})
        candidates.append(did)
        if title in support_titles: supports.append(did)
    question_docs[q["id"]] = sorted(set(candidates))
    gold_support_docs[q["id"]] = sorted(set(supports))
assert docs, "文档为空"
atomic_json(ROOT / "corpus.json", dict(docs=docs, question_docs=question_docs))
# gold 标注与 corpus.json 分开存储，后续索引/检索完全不读取
atomic_json(ROOT / "gold_only_for_evaluation.json", gold_support_docs)

doc_table = pd.DataFrame([dict(doc_id=d, title=p["title"],
    chars=len(p["text"]), preview=p["text"][:90]) for d,p in docs.items()])
print("问题数：", len(questions), "唯一文档数：", len(docs))
display(doc_table.head(12))
fig = px.histogram(doc_table, x="chars", nbins=20, title="候选文档长度分布（字符数）")
fig.update_layout(height=320)
fig.show()

问题数： 20 唯一文档数： 200


,doc_id,title,chars,preview
0,ee3f77178c395664e9e4,Robin Barton,680,Robin Barton (born 5 November 1958) is a Briti...
1,e66e3768055c5918df3b,Funny Bones,478,Funny Bones is a 1995 British-American comedy-...
2,eab198086fce65d88781,Ivan Dragomiloff,208,"Ivan Dragomiloff is a fictional character, the..."
3,bb9945cb5ab48df85f2a,Oliver Reed,590,Robert Oliver Reed (13 February 1938 – 2 May 1...
4,4f44e2bddef4decbf9f4,Harry Flashman,443,Sir Harry Paget Flashman is a fictional charac...
5,1377ef4dedadd8193f4e,Royal Flash (film),357,Royal Flash is a 1975 film based on George Mac...
6,ef3abbe53541a7e85146,Royal Flash,196,Royal Flash is a 1970 novel by George MacDonal...
7,f31d370690ecda663912,The Duke of Hamilton,237,The Duke of Hamilton was one of the oldest pub...
8,c439b304cae6aa8cbb0c,Lion of the Desert,633,Lion of the Desert is a 1981 Libyan historical...
9,17d5c025879f16b7f99e,Otto von Bismarck,1014,"Otto Eduard Leopold, Prince of Bismarck, Duke ..."


## 第 4 步｜让 GPT-4o-mini 从每篇文本抽取实体和关系

这是最重要的“原始证据 → 结构化图谱”过程。**每篇文档单独抽取**，不附带问题和 gold。每个结果存成 JSON，重跑跳过已完成文档。可在下一个单元格检查实体类型、关系和 JSON 原文。

In [25]:
EXTRACT_SYS = """You are a precise information extraction system for a factual knowledge graph.
Only use explicitly supported claims from the provided document. Return one valid JSON object:
{"entities":[{"name":"exact name","type":"PERSON|ORG|PLACE|EVENT|WORK|OTHER","description":"short evidence-grounded description"}],
"relationships":[{"source":"exact entity name","target":"exact entity name","description":"explicit factual relationship"}]}.
Include up to 12 named entities and up to 15 nontrivial relations; do not invent facts.
All relationship endpoints MUST be entities you included. No explanation outside JSON."""

def norm_entity(name):
    return " ".join(str(name).casefold().split())

def extract_doc(doc):
    prompt = f"Title: {doc['title']}\nText:\n{doc['text'][:12000]}"
    response = client.chat.completions.create(
        model=EXTRACT_MODEL,
        temperature=0,
        max_tokens=MAX_EXTRACT_OUTPUT,
        response_format={"type": "json_object"},
        messages=[{"role":"system", "content":EXTRACT_SYS},
                  {"role":"user", "content":prompt}])
    obj = json.loads(response.choices[0].message.content)
    entities, relationships, names = [], [], {}
    for entity in obj.get("entities", []):
        if not isinstance(entity, dict): continue
        name = str(entity.get("name", "")).strip()
        if not name or len(name) > 130: continue
        key = norm_entity(name)
        if key in names: continue
        names[key] = name
        entities.append(dict(name=name, type=str(entity.get("type", "OTHER"))[:35],
                             description=str(entity.get("description", ""))[:600]))
    for relation in obj.get("relationships", []):
        if not isinstance(relation, dict): continue
        s, t = norm_entity(relation.get("source", "")), norm_entity(relation.get("target", ""))
        if s not in names or t not in names or s == t: continue
        relationships.append(dict(source=names[s], target=names[t],
                                  description=str(relation.get("description", ""))[:500]))
    return dict(entities=entities, relationships=relationships,
                usage=usage_dict(response.usage), raw=obj)

def extract_and_save(did):
    path = ROOT / "extractions" / f"{did}.json"
    if path.exists(): return "cache"
    for attempt in range(3):
        try:
            result = extract_doc(docs[did])
            atomic_json(path, result)
            return "created"
        except Exception:
            if attempt == 2: raise
            time.sleep(2 ** attempt)

def run_extraction():
    pending = [did for did in docs if not (ROOT / "extractions" / f"{did}.json").exists()]
    print(f"已有缓存 {len(docs)-len(pending)} / {len(docs)}；本次需抽取 {len(pending)} 篇")
    failed = {}
    with ThreadPoolExecutor(max_workers=EXTRACT_WORKERS) as pool:
        jobs = {pool.submit(extract_and_save, did): did for did in pending}
        for task in tqdm(as_completed(jobs), total=len(jobs), desc="GPT 实体/关系抽取"):
            did = jobs[task]
            try: task.result()
            except Exception as exc: failed[did] = repr(exc)
    atomic_json(ROOT / "extractions" / "_failures.json", failed)
    if failed: raise RuntimeError(f"实体抽取失败 {len(failed)} 篇，检查 _failures.json 后重跑本单元格")
    print("完成：", len(docs), "篇；无失败。")

run_extraction()

已有缓存 0 / 200；本次需抽取 200 篇


GPT 实体/关系抽取:   0%|          | 0/200 [00:00<?, ?it/s]

完成： 200 篇；无失败。


### 看一看真实抽取结果（不是示例假数据）

这一步展示第一篇文档的 `entities`、`relationships` 和 GPT 返回的原始 JSON，并统计整批文本的实体/关系数量。若某篇文档没有抽到关系，可能是原文事实过少，也可能是抽取质量问题。

In [26]:
extractions = {}
entity_rows, relation_rows, token_rows = [], [], []
for did in docs:
    path = ROOT / "extractions" / f"{did}.json"
    if not path.exists(): raise FileNotFoundError(path)
    extraction = json.loads(path.read_text(encoding="utf-8"))
    extractions[did] = extraction
    token_rows.append(dict(doc_id=did, **extraction["usage"]))
    for e in extraction["entities"]:
        entity_rows.append(dict(doc_id=did, **e))
    for e in extraction["relationships"]:
        relation_rows.append(dict(doc_id=did, **e))

entity_df = pd.DataFrame(entity_rows)
relation_df = pd.DataFrame(relation_rows)
print("实体提及：", len(entity_df), "关系提及：", len(relation_df))
print("抽取阶段实际 GPT tokens：", pd.DataFrame(token_rows)["total_tokens"].sum())
display(entity_df.head(15))
display(relation_df.head(15))
first_doc = next(iter(docs))
print("首篇文档原始模型输出 JSON:")
print(json.dumps(extractions[first_doc]["raw"], indent=2, ensure_ascii=False)[:4500])
if not entity_df.empty:
    counts = entity_df["type"].value_counts().reset_index()
    counts.columns = ["type", "count"]
    px.bar(counts, x="type", y="count", title="抽取实体的类型分布", height=340).show()

实体提及： 1749 关系提及： 1726
抽取阶段实际 GPT tokens： 193183


,doc_id,name,type,description
0,ee3f77178c395664e9e4,Robin Barton,PERSON,British art dealer dealing primarily with Bank...
1,ee3f77178c395664e9e4,Banksy,PERSON,Famous street artist known for his provocative...
2,ee3f77178c395664e9e4,Exeter College of Art and Design,ORG,Educational institution where Robin Barton stu...
3,ee3f77178c395664e9e4,Russell Young,PERSON,Individual with whom Robin Barton had his firs...
4,ee3f77178c395664e9e4,London,PLACE,City where Robin Barton moved in 1980 to pursu...
5,ee3f77178c395664e9e4,Sounds,ORG,Music publication for which Robin Barton worke...
6,ee3f77178c395664e9e4,NME,ORG,Music publication where Robin Barton contribut...
7,ee3f77178c395664e9e4,Blitz,ORG,Fashion publication that employed Robin Barton...
8,ee3f77178c395664e9e4,The Face,ORG,Pioneering publication for which Robin Barton ...
9,ee3f77178c395664e9e4,Independent Magazine,ORG,Publication where Robin Barton regularly worke...


,doc_id,source,target,description
0,ee3f77178c395664e9e4,Robin Barton,Banksy,Robin Barton is an art dealer dealing primaril...
1,ee3f77178c395664e9e4,Robin Barton,Exeter College of Art and Design,Robin Barton studied photography and graphic d...
2,ee3f77178c395664e9e4,Robin Barton,Russell Young,Robin Barton had his first encounter with Russ...
3,ee3f77178c395664e9e4,Robin Barton,London,Robin Barton moved to London in 1980.
4,ee3f77178c395664e9e4,Robin Barton,Sounds,Robin Barton worked as a freelance photographe...
5,ee3f77178c395664e9e4,Robin Barton,NME,Robin Barton worked as a freelance photographe...
6,ee3f77178c395664e9e4,Robin Barton,Blitz,Robin Barton worked as a freelance photographe...
7,ee3f77178c395664e9e4,Robin Barton,The Face,Robin Barton worked as a freelance photographe...
8,ee3f77178c395664e9e4,Robin Barton,Independent Magazine,Robin Barton regularly worked for Independent ...
9,ee3f77178c395664e9e4,Robin Barton,Sir Alec Guinness,Robin Barton photographed Sir Alec Guinness.


首篇文档原始模型输出 JSON:
{
  "entities": [
    {
      "name": "Robin Barton",
      "type": "PERSON",
      "description": "British art dealer dealing primarily with Banksy's."
    },
    {
      "name": "Banksy",
      "type": "PERSON",
      "description": "Famous street artist known for his provocative and politically charged artwork."
    },
    {
      "name": "Exeter College of Art and Design",
      "type": "ORG",
      "description": "Educational institution where Robin Barton studied photography and graphic design."
    },
    {
      "name": "Russell Young",
      "type": "PERSON",
      "description": "Individual with whom Robin Barton had his first encounter during his studies."
    },
    {
      "name": "London",
      "type": "PLACE",
      "description": "City where Robin Barton moved in 1980 to pursue his career."
    },
    {
      "name": "Sounds",
      "type": "ORG",
      "description": "Music publication for which Robin Barton worked as a freelance photographer."
    },

## 第 5 步｜手工合并知识图谱：实体成为节点，关系成为边

同名实体统一使用标准化键（大小写和空格规整）；同一对实体多次共现时合并关系并累积来源文档。**注意同名不保证同一实体**：这是教学代码中的简化实体对齐策略，会在研究报告中注明。

In [27]:
nodes, edge_map = {}, {}
for did, result in extractions.items():
    for e in result["entities"]:
        key = norm_entity(e["name"])
        if not key: continue
        node = nodes.setdefault(key, dict(key=key, name=e["name"], type=e["type"],
            descriptions=[], doc_ids=set()))
        if e["description"]: node["descriptions"].append(e["description"])
        node["doc_ids"].add(did)
    for r in result["relationships"]:
        a, b = norm_entity(r["source"]), norm_entity(r["target"])
        if a not in nodes or b not in nodes or a == b: continue
        pair = tuple(sorted([a, b]))
        edge = edge_map.setdefault(pair, dict(source=pair[0], target=pair[1],
                        descriptions=[], doc_ids=set()))
        if r["description"]: edge["descriptions"].append(r["description"])
        edge["doc_ids"].add(did)

G = nx.Graph()
for k, v in nodes.items():
    v["doc_ids"] = sorted(v["doc_ids"])
    v["description"] = "; ".join(dict.fromkeys(v.pop("descriptions")))[:900]
    G.add_node(k, name=v["name"], type=v["type"], description=v["description"],
               doc_ids=v["doc_ids"])
for (a,b), e in edge_map.items():
    e["doc_ids"] = sorted(e["doc_ids"])
    e["description"] = "; ".join(dict.fromkeys(e.pop("descriptions")))[:900]
    e["weight"] = len(e["doc_ids"])
    G.add_edge(a, b, weight=e["weight"], description=e["description"], doc_ids=e["doc_ids"])
if not len(G): raise RuntimeError("无实体节点，请检查第 4 步的抽取结果")

node_table = pd.DataFrame([dict(name=nodes[n]["name"], type=nodes[n]["type"],
    degree=G.degree(n), document_count=len(nodes[n]["doc_ids"]),
    description=nodes[n]["description"][:100]) for n in G.nodes])
edge_table = pd.DataFrame([dict(source=nodes[u]["name"], target=nodes[v]["name"],
    weight=G[u][v]["weight"], relation=G[u][v]["description"][:120]) for u,v in G.edges])
print(f"知识图谱：{G.number_of_nodes()} 节点 / {G.number_of_edges()} 边 / {nx.number_connected_components(G)} 个连通分量")
display(node_table.sort_values("degree", ascending=False).head(12))
display(edge_table.head(12))

# 导出包含孤立节点的完整 GraphML；复杂来源列表仍写在后续 JSON 中
export_graph=nx.Graph()
for n in G.nodes:
    export_graph.add_node(n, label=nodes[n]["name"], entity_type=nodes[n]["type"],
                          document_count=len(nodes[n]["doc_ids"]))
for a,b in G.edges:
    export_graph.add_edge(a,b,weight=float(G[a][b]["weight"]),
                          description=G[a][b]["description"])
nx.write_graphml(export_graph, ROOT / "graph_edges.graphml")

知识图谱：1407 节点 / 1604 边 / 63 个连通分量


,name,type,degree,document_count,description
1182,Crazy Love,WORK,24,8,A 2009 album by Michael Bublé that includes th...
469,Gunfight at the O.K. Corral,EVENT,19,10,A 30-second gunfight that occurred on October ...
87,University of Utah Singers,ORG,18,1,The premier choral ensemble at the University ...
402,Florey,OTHER,15,1,An electoral district of the House of Assembly...
1352,Gasherbrum II,PLACE,15,8,Another mountain in the Karakoram range climbe...
199,Win,WORK,15,1,A romance thriller trilingual film directed by...
0,Robin Barton,PERSON,15,1,British art dealer dealing primarily with Bank...
345,Para Hills,PLACE,14,6,"A residential suburb of Adelaide, South Austra..."
922,Charles D. Sherwood,PERSON,14,1,"A Minnesota politician, youngest Speaker of th..."
866,Charles Wilson Bagnal,PERSON,13,1,"Lieutenant General in the United States Army, ..."


,source,target,weight,relation
0,Robin Barton,Banksy,1,Robin Barton is an art dealer dealing primaril...
1,Robin Barton,Exeter College of Art and Design,1,Robin Barton studied photography and graphic d...
2,Robin Barton,Russell Young,1,Robin Barton had his first encounter with Russ...
3,Robin Barton,London,1,Robin Barton moved to London in 1980.
4,Robin Barton,Sounds,1,Robin Barton worked as a freelance photographe...
5,Robin Barton,NME,1,Robin Barton worked as a freelance photographe...
6,Robin Barton,Blitz,1,Robin Barton worked as a freelance photographe...
7,Robin Barton,The Face,1,Robin Barton worked as a freelance photographe...
8,Robin Barton,Independent Magazine,1,Robin Barton regularly worked for Independent ...
9,Robin Barton,Sir Alec Guinness,1,Robin Barton photographed Sir Alec Guinness.


## 第 6 步｜使用 Leiden 做社区划分

在图上寻找关系紧密的实体群。这里采用**单层 Leiden**，不是官方 GraphRAG 的多层社区树；你可以直观看各社区包含哪些实体、相关边和来源文档。

下面**图像是从第 5 步真正抽取出来的数据绘制的**。如果实体很多，为了保持浏览器流畅，只显示最多 `VIS_MAX_NODES` 个高连接度节点；Leiden 本身仍在完整图上运行。

In [28]:
vertex_keys = list(G.nodes)
vertex_idx = {k: i for i, k in enumerate(vertex_keys)}
ig_graph = ig.Graph(n=len(vertex_keys),
    edges=[(vertex_idx[u], vertex_idx[v]) for u,v in G.edges], directed=False)
ig_graph.vs["name"] = vertex_keys
if G.number_of_edges():
    ig_graph.es["weight"] = [G[u][v]["weight"] for u,v in G.edges]
    partition = leidenalg.find_partition(ig_graph, leidenalg.RBConfigurationVertexPartition,
        weights="weight", resolution_parameter=LEIDEN_RESOLUTION, seed=SEED)
    membership = list(partition.membership)
else:
    membership = list(range(len(vertex_keys)))
community_of = {k: int(membership[i]) for i,k in enumerate(vertex_keys)}
communities = {}
for k,cid in community_of.items():
    communities.setdefault(cid, {"id": cid, "entity_keys": [], "doc_ids": set()})
    communities[cid]["entity_keys"].append(k)
    communities[cid]["doc_ids"].update(nodes[k]["doc_ids"])
for cid,c in communities.items():
    c["doc_ids"] = sorted(c["doc_ids"])
    c["n_entities"] = len(c["entity_keys"])
    c["n_internal_edges"] = sum(community_of[u]==cid and community_of[v]==cid for u,v in G.edges)

community_stats = pd.DataFrame([dict(community_id=c, entities=v["n_entities"],
    edges=v["n_internal_edges"], source_documents=len(v["doc_ids"])) for c,v in communities.items()])
community_stats = community_stats.sort_values("entities", ascending=False)
print("Leiden 检测出社区：", len(communities))
display(community_stats.head(15))
px.bar(community_stats.head(25), x="community_id", y="entities", color="entities",
       title="各社区实体数量（最多展示前 25 个）", height=370).show()

atomic_json(ROOT / "graph_index.json", dict(
    nodes=nodes,
    edges=[dict(source=u, target=v, **G[u][v]) for u,v in G.edges],
    community_of=community_of, communities={str(k):v for k,v in communities.items()},
    method="single-level Leiden on extracted-entity graph"))

Leiden 检测出社区： 80


,community_id,entities,edges,source_documents
5,0,71,88,16
10,1,68,73,15
20,2,62,76,9
33,3,60,63,9
7,4,56,56,6
1,5,55,64,12
3,6,53,62,10
0,7,53,61,9
18,8,53,64,10
23,9,52,75,10


### 交互图｜实体关系网络（社区着色）

- 圆圈 = 实体；线 = 关系；圆圈越大表示度数越大。
- 同色通常表示属于同一 Leiden 社区。
- 鼠标悬停可查看实体名称、社区、文档数量。下面的 `draw_knowledge_graph` 函数还能用于单独查看某个社区或 Local 的检索子图。

In [29]:
from plotly.colors import qualitative

def draw_knowledge_graph(graph, community_colors=None, title="Knowledge graph",
                         limit=VIS_MAX_NODES, output_path=None):
    if graph.number_of_nodes() == 0:
        print("子图为空")
        return None
    ranking = sorted(graph.nodes, key=lambda n: (graph.degree(n), n), reverse=True)
    show = ranking[:limit]
    sub = graph.subgraph(show).copy()
    positions = nx.spring_layout(sub, seed=SEED, weight="weight", iterations=65)
    ex, ey = [], []
    for a,b in sub.edges:
        x0,y0 = positions[a]; x1,y1 = positions[b]
        ex.extend([x0,x1,None]); ey.extend([y0,y1,None])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ex, y=ey, mode="lines", hoverinfo="skip",
        line=dict(width=.8, color="rgba(125,125,125,.35)"), name="Relations", showlegend=False))
    palette = qualitative.Plotly + qualitative.Dark24 + qualitative.Set3
    colors = community_colors if community_colors is not None else community_of
    by_color = defaultdict(list)
    for n in sub.nodes:
        by_color[colors.get(n, -1)].append(n)
    for cid, group in sorted(by_color.items(), key=lambda x: (x[0] < 0, x[0])):
        ids = [int(cid)] if isinstance(cid, (int,np.integer)) else [0]
        cindex = ids[0]
        color = "#8c8c8c" if cindex < 0 else palette[cindex % len(palette)]
        x=[positions[n][0] for n in group]
        y=[positions[n][1] for n in group]
        hover=[f"{nodes[n]['name']}<br>Community: {community_of.get(n,'?')}"
               f"<br>Degree (subgraph): {sub.degree(n)}<br>Source docs: {len(nodes[n]['doc_ids'])}"
               for n in group]
        size=[min(33, 10+4*np.sqrt(sub.degree(n)+1)) for n in group]
        fig.add_trace(go.Scatter(x=x,y=y,mode="markers+text", name=f"Community {cid}",
            text=[nodes[n]["name"] if sub.degree(n)>=3 else "" for n in group],
            textposition="top center", textfont=dict(size=9),
            marker=dict(size=size,color=color,line=dict(width=.5,color="white")),
            hovertext=hover,hoverinfo="text"))
    fig.update_layout(title=f"{title}  (shown {len(sub.nodes)}/{len(graph)} nodes)",
        height=690, template="plotly_white", showlegend=len(by_color)<=16,
        xaxis=dict(visible=False),yaxis=dict(visible=False),
        margin=dict(l=20,r=20,t=55,b=20))
    if output_path:
        fig.write_html(str(output_path), include_plotlyjs="cdn")
    fig.show()
    return fig

knowledge_fig = draw_knowledge_graph(G, title="HotpotQA entity graph · colored by Leiden community",
    output_path=ROOT / "figures" / "entity_communities.html")
print("交互 HTML 已保存到：", ROOT / "figures" / "entity_communities.html")

交互 HTML 已保存到： /content/drive/MyDrive/GraphRAG_HotpotQA_Tutorial/hotpot_n20_seed42_9c3a52009c3d/figures/entity_communities.html


## 第 7 步｜根据图结构生成 Community Reports

我们将一个社区内的**实体描述、内部关系及证据标题**交给 GPT-4o-mini，让它生成可检查的社区报告 JSON：`title`、`summary`、`key_findings`。这些报告就是 Global Search 的知识来源。报告包含社区 ID 和来源文档映射，不使用标准答案。

In [30]:
REPORT_SYS = """You write concise and faithful community reports for a knowledge graph.
Summarize only the supplied entity descriptions and edges, no extra facts. Return JSON object:
{"title":"community title", "summary":"120-220 words at most", "key_findings":["finding 1", "finding 2"]}.
If the community contains very little information, say so; never hallucinate."""

def report_context(cid):
    comm = communities[cid]
    ks = sorted(comm["entity_keys"], key=lambda k: G.degree(k), reverse=True)
    lines = [f"Community {cid}. Entity count {len(ks)}", "ENTITIES:"]
    for k in ks[:65]:
        lines.append(f"- {nodes[k]['name']} ({nodes[k]['type']}): {nodes[k]['description'][:300]}")
    lines.append("INTRA-COMMUNITY RELATIONS:")
    n_edges = 0
    for a,b in G.edges:
        if community_of[a] == cid and community_of[b] == cid:
            lines.append(f"- {nodes[a]['name']} -- {nodes[b]['name']}: {G[a][b]['description'][:250]}")
            n_edges += 1
            if n_edges >= 90: break
    lines.append("DOCUMENT TITLES (only as provenance, not full text):")
    for did in comm["doc_ids"][:20]: lines.append("- " + docs[did]["title"])
    return "\n".join(lines)[:26000]

def report_one(cid):
    path = ROOT / "reports" / f"community_{cid}.json"
    if path.exists(): return "cache"
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=REPORT_MODEL, temperature=0, max_tokens=MAX_REPORT_OUTPUT,
                response_format={"type":"json_object"},
                messages=[{"role":"system","content":REPORT_SYS},
                          {"role":"user","content":report_context(cid)}])
            info = json.loads(response.choices[0].message.content)
            findings = info.get("key_findings") or []
            if not isinstance(findings,list): findings=[str(findings)]
            record = dict(community_id=cid, title=str(info.get("title") or f"Community {cid}"),
                summary=str(info.get("summary") or ""), key_findings=[str(x) for x in findings][:12],
                doc_ids=communities[cid]["doc_ids"], n_entities=communities[cid]["n_entities"],
                usage=usage_dict(response.usage))
            if not record["summary"]: raise ValueError("Empty report summary")
            atomic_json(path,record)
            return "created"
        except Exception:
            if attempt==2: raise
            time.sleep(2**attempt)

def generate_reports():
    pending=[cid for cid in communities if not (ROOT/"reports"/f"community_{cid}.json").exists()]
    print(f"已有报告：{len(communities)-len(pending)}；本次需生成：{len(pending)}")
    failures={}
    with ThreadPoolExecutor(max_workers=REPORT_WORKERS) as pool:
        fut={pool.submit(report_one,cid):cid for cid in pending}
        for t in tqdm(as_completed(fut), total=len(fut), desc="GPT 社区报告"):
            try:t.result()
            except Exception as exc:failures[fut[t]]=repr(exc)
    atomic_json(ROOT/"reports"/"_failures.json", failures)
    if failures:raise RuntimeError(f"有 {len(failures)} 个社区报告失败，可重跑本单元格")

generate_reports()
reports = {cid:json.loads((ROOT/"reports"/f"community_{cid}.json").read_text(encoding="utf-8"))
           for cid in communities}

def report_text(r):
    return r["title"] + "\n" + r["summary"] + "\n" + "\n".join("- "+x for x in r["key_findings"])

report_table = pd.DataFrame([dict(community_id=cid, title=r["title"],
    n_entities=r["n_entities"], num_findings=len(r["key_findings"]),
    summary_preview=r["summary"][:110]) for cid,r in reports.items()])
print("已生成报告：",len(reports),"个")
display(report_table.sort_values("n_entities", ascending=False).head(12))

已有报告：0；本次需生成：80


GPT 社区报告:   0%|          | 0/80 [00:00<?, ?it/s]

已生成报告： 80 个


,community_id,title,n_entities,num_findings,summary_preview
5,0,Biathlon and Cultural Performances Community,71,2,This community encompasses a diverse range of ...
10,1,Community Overview of Twisted Fortune and Rela...,68,2,This community encompasses a diverse range of ...
20,2,Kentucky Pioneers and Historical Events,62,2,This community encompasses a range of historic...
33,3,Eddie Murphy and His Works,60,2,This community focuses on the influential Amer...
7,4,Community 4,56,2,This community encompasses a diverse array of ...
1,5,Christmas Literature and Adaptations,55,2,This community encompasses a variety of entiti...
3,6,Military Leadership and Historical Context in ...,53,2,Community 6 encompasses a diverse array of ent...
0,7,Community 7,53,2,"Community 7 consists of 53 entities, primarily..."
18,8,Community 8 Overview,53,2,Community 8 encompasses a diverse range of ent...
23,9,Community 9 Overview,52,2,Community 9 encompasses various suburbs and el...


### 深入查看一个社区：报告正文 ↔ 真实实体/关系 ↔ 来源文档

默认查看**节点数最多的社区**。你可以把 `CID_TO_INSPECT` 换成上表任何一个社区编号，重复执行这个单元格；不会重新调用模型。

In [31]:
CID_TO_INSPECT = int(community_stats.iloc[0]["community_id"])
r = reports[CID_TO_INSPECT]
c = communities[CID_TO_INSPECT]
print("Community ID:", CID_TO_INSPECT, "| 实体:", c["n_entities"], "| 来源文档:",len(c["doc_ids"]))
md = f"### {r['title']}\n\n{r['summary']}\n\n**Key findings**\n" + "\n".join("- "+f for f in r["key_findings"])
display(Markdown(md))
print("\n此社区包含的实体（前 20 个）:")
display(pd.DataFrame([dict(name=nodes[k]["name"], type=nodes[k]["type"],
     degree=G.degree(k)) for k in c["entity_keys"]]).sort_values("degree",ascending=False).head(20))
print("来源文档（前 12 个）：")
display(pd.DataFrame([dict(doc_id=did,title=docs[did]["title"]) for did in c["doc_ids"][:12]]))
subgraph=G.subgraph(c["entity_keys"]).copy()
draw_knowledge_graph(subgraph,title=f"Community {CID_TO_INSPECT} · extracted knowledge",limit=70)

Community ID: 0 | 实体: 71 | 来源文档: 16


### Biathlon and Cultural Performances Community

This community encompasses a diverse range of entities related to biathlon events, notable athletes, and cultural performances by the University of Utah Singers. The University of Utah Singers, a premier choral ensemble until 2010, performed in various countries including Germany, Austria, and France, winning multiple competitions such as the Grand Prize at the Florilège Vocal de Tours International Choir Competition in 2005. The biathlon events highlighted include the 2009–10 Biathlon World Cup series, featuring individual and relay competitions for both men and women, with significant locations like Östersund and Vancouver. Prominent athletes such as Ole Einar Bjørndalen, the most medaled Olympian in Winter Olympic history, and Martin Fourcade, a leading biathlete from France, are also featured. The community reflects the intersection of sports and cultural achievements across Europe.

**Key findings**
- The University of Utah Singers achieved notable success in international choral competitions.
- The 2009–10 Biathlon World Cup series included significant events and locations, showcasing top athletes in the sport.


此社区包含的实体（前 20 个）:


,name,type,degree
11,University of Utah Singers,ORG,18
1,Germany,PLACE,9
53,Ole Einar Bjørndalen,PERSON,8
0,Otto von Bismarck,PERSON,7
39,Östersund,PLACE,7
38,2009–10 Biathlon World Cup – Individual Women,EVENT,6
59,2009–10 Biathlon World Cup – Individual Men,EVENT,5
65,2009–10 Biathlon World Cup – Relay Women,EVENT,5
40,Vancouver,PLACE,5
63,2008-09 Biathlon World Cup,EVENT,4


来源文档（前 12 个）：


,doc_id,title
0,03f4ae0f974723c70d21,Kurt Weill
1,17d5c025879f16b7f99e,Otto von Bismarck
2,1bd5c350cab945b8d1f5,2009–10 Biathlon World Cup – Pursuit Women
3,1f4844b2dd81e258f7d2,2008–09 Biathlon World Cup – Pursuit Men
4,32defbb7cbaa98eccd0d,Rob Jackson
5,433bbc05ca5a4148f582,2010–11 Biathlon World Cup – Pursuit Men
6,557b630094731248301a,Sixth United States Army Group
7,5c612c64ac1d11f0a991,South German Offensive
8,6ab62e6eb6f5cc7692a3,2009–10 Biathlon World Cup – Pursuit Men
9,85e7a4ad85263fb14789,2009–10 Biathlon World Cup – Individual Men


## 第 8 步｜为实体、原始文档、社区报告建立 Embedding

同一个 `text-embedding-3-small` 会把三类文本映射到同一向量空间：

1. `entity name + description`：Local Search 找种子实体。
2. `title + document text`：Local Search 在候选文档中再排序。
3. `Community Report`：Global Search 找相关社区。

所有向量会保存到 Drive（`.npz`）。检索时**不读取 gold answer 或 supporting facts**。

In [32]:
embedding_usage = {}

def embed_texts(collection_name, texts, batch_size=80):
    texts=[str(x) for x in texts]
    fingerprint=sha(json.dumps({"model":EMBED_MODEL,"texts":texts},ensure_ascii=False))
    safe_name=re.sub("[^A-Za-z0-9_-]","_",collection_name)[:50]
    path=ROOT/"embeddings"/f"{safe_name}_{fingerprint}.npz"
    if path.exists():
        with np.load(path,allow_pickle=False) as f:
            vec=f["matrix"].astype("float32")
        if vec.shape[0]!=len(texts): raise RuntimeError("Embedding cache row count mismatch")
        embedding_usage[collection_name]={"cached":True,"total_tokens_new":0}
        return vec
    if not texts: raise ValueError("Cannot embed empty list")
    parts=[]; used=0
    for start in tqdm(range(0,len(texts),batch_size),desc="Embedding " + collection_name):
        resp=client.embeddings.create(model=EMBED_MODEL,input=texts[start:start+batch_size])
        chunk=[x.embedding for x in sorted(resp.data,key=lambda item:item.index)]
        parts.extend(chunk)
        used+=int(getattr(resp.usage,"total_tokens",0) or 0)
    matrix=np.asarray(parts,dtype="float32")
    matrix/=np.maximum(np.linalg.norm(matrix,axis=1,keepdims=True),1e-12)
    with open(str(path)+".tmp","wb") as f:
        np.savez_compressed(f,matrix=matrix)
    os.replace(str(path)+".tmp",path)
    embedding_usage[collection_name]={"cached":False,"total_tokens_new":used}
    return matrix

entity_keys = list(nodes)
document_ids = list(docs)
report_ids = sorted(reports)
entity_vectors = embed_texts("entity_descriptions",[
    nodes[k]["name"]+". "+nodes[k]["description"] for k in entity_keys])
document_vectors = embed_texts("original_documents",[
    docs[d]["title"]+"\n"+docs[d]["text"][:5000] for d in document_ids])
report_vectors = embed_texts("community_reports",[report_text(reports[cid]) for cid in report_ids])
entity_positions={k:i for i,k in enumerate(entity_keys)}
document_positions={d:i for i,d in enumerate(document_ids)}
report_positions={cid:i for i,cid in enumerate(report_ids)}
print("实体向量形状:",entity_vectors.shape,
      "文档向量形状:",document_vectors.shape,
      "报告向量形状:",report_vectors.shape)
display(pd.DataFrame([dict(collection=k, **v) for k,v in embedding_usage.items()]))

def embed_question(text):
    return embed_texts("question_"+sha(text),[text])[0]

def rank_vectors(q_vector, matrix, ids, k=8):
    sims = matrix @ q_vector
    order=np.argsort(-sims)[:min(k,len(ids))]
    return [(ids[i],float(sims[i])) for i in order]

# 展示例题分别召回了哪些实体、哪些文档、哪些社区
q_example=questions[0]["question"]
qvec=embed_question(q_example)
top_entities=rank_vectors(qvec,entity_vectors,entity_keys,10)
top_documents=rank_vectors(qvec,document_vectors,document_ids,8)
top_reports=rank_vectors(qvec,report_vectors,report_ids,8)
print("问题：",q_example)
print("\n向量最相似的实体：")
display(pd.DataFrame([dict(entity=nodes[k]["name"], similarity=round(s,3),
    community=community_of[k]) for k,s in top_entities]))
print("\n向量最相似的候选文档：")
display(pd.DataFrame([dict(title=docs[d]["title"],doc_id=d,similarity=round(s,3))
    for d,s in top_documents]))
print("\n向量最相似的 Community Reports：")
display(pd.DataFrame([dict(community_id=cid, title=reports[cid]["title"],
    similarity=round(s,3)) for cid,s in top_reports]))

Embedding entity_descriptions:   0%|          | 0/18 [00:00<?, ?it/s]

Embedding original_documents:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding community_reports:   0%|          | 0/1 [00:00<?, ?it/s]

实体向量形状: (1407, 1536) 文档向量形状: (200, 1536) 报告向量形状: (80, 1536)


,collection,cached,total_tokens_new
0,entity_descriptions,False,31164
1,original_documents,False,25945
2,community_reports,False,14931


...

,entity,similarity,community
0,Robert Oliver Reed,0.655,20
1,Oliver Reed,0.622,7
2,Oliver!,0.619,20
3,Malcolm McDowell,0.578,7
4,Alan Bates,0.553,7
5,Richard Lester,0.539,7
6,BAFTA Award for Best Actor in a Supporting Role,0.534,20
7,Tommy,0.525,20
8,Hannibal Brooks,0.525,20
9,1975,0.522,7



向量最相似的候选文档：


,title,doc_id,similarity
0,Royal Flash (film),1377ef4dedadd8193f4e,0.640
1,Oliver Reed,bb9945cb5ab48df85f2a,0.631
2,Royal Flash,ef3abbe53541a7e85146,0.478
3,Harry Flashman,4f44e2bddef4decbf9f4,0.456
4,The Duke of Hamilton,f31d370690ecda663912,0.386
5,Ivan Dragomiloff,eab198086fce65d88781,0.375
6,Lion of the Desert,c439b304cae6aa8cbb0c,0.351
7,Funny Bones,e66e3768055c5918df3b,0.309



向量最相似的 Community Reports：


,community_id,title,similarity
0,20,Community 20: Oliver Reed and Funny Bones,0.496
1,7,Community 7,0.407
2,42,Community 42,0.277
3,32,Alfred Hurley Community,0.272
4,29,Community 29: John Braddocke and His Connections,0.256
5,1,Community Overview of Twisted Fortune and Rela...,0.237
6,39,Community 39,0.229
7,37,Operation Quicksilver and Deception in WWII,0.227


## 第 9 步｜手写 Local Search：实体种子 → 图邻居 → 文档重排

下面先**只运行检索，不生成答案**，让你看到 Local Search 为什么拿到这些文档。算法：问题匹配实体 Embedding → 种子节点及其一跳邻居 → 按图来源关系收集文档 → 文档 Embedding 重排序 → 合并节点、关系、社区报告和原文。

In [33]:
def local_retrieve(question):
    qvec=embed_question(question)
    seeds=rank_vectors(qvec,entity_vectors,entity_keys,LOCAL_TOP_ENTITIES)
    seed_keys=[k for k,_ in seeds]
    related=set(seed_keys)
    for k in seed_keys: related.update(G.neighbors(k))
    touching=[(a,b) for a,b in G.edges if a in seed_keys or b in seed_keys]
    raw_scores=defaultdict(float)
    for k in related:
        for did in nodes[k]["doc_ids"]:
            raw_scores[did]+=2.0 if k in seed_keys else .4
    candidate_ids=sorted(raw_scores,key=lambda did:(-raw_scores[did],did))[:max(LOCAL_MAX_DOCS*4,LOCAL_MAX_DOCS)]
    if not candidate_ids: candidate_ids=document_ids
    doc_ranking=sorted(((d,float(document_vectors[document_positions[d]] @ qvec)) for d in candidate_ids),
                      key=lambda item:-item[1])[:LOCAL_MAX_DOCS]
    picked_docs=[d for d,_ in doc_ranking]
    relevant_community_ids=sorted(set(community_of[k] for k in seed_keys),
                        key=lambda cid:-sum(community_of[k]==cid for k in seed_keys))[:2]
    parts=["SELECTED GRAPH ENTITIES:"]
    for k in seed_keys:
        parts.append(f"- {nodes[k]['name']}: {nodes[k]['description'][:300]}")
    parts.append("RELATIONS TOUCHING SEED ENTITIES:")
    for a,b in touching[:22]:
        parts.append(f"- {nodes[a]['name']} <-> {nodes[b]['name']}: {G[a][b]['description'][:260]}")
    parts.append("RELATED COMMUNITY SUMMARIES:")
    for cid in relevant_community_ids:
        parts.append(f"[Community {cid}] {report_text(reports[cid])[:1200]}")
    parts.append("ORIGINAL TEXT EVIDENCE (source doc ids):")
    for i,did in enumerate(picked_docs):
        d=docs[did]
        parts.append(f"[Doc {i+1}, id={did}, title={d['title']}]\n{d['text'][:2100]}")
    return dict(seed_keys=seed_keys,seed_scores=seeds, related_nodes=sorted(related),
        touching_edges=touching, doc_ids=picked_docs, doc_scores=doc_ranking,
        report_ids=relevant_community_ids, context="\n\n".join(parts))

example_local=local_retrieve(q_example)
print("Local 的种子实体：",[nodes[k]["name"] for k in example_local["seed_keys"]])
print("一跳扩展后实体数量：",len(example_local["related_nodes"]))
print("涉及种子的关系数量：",len(example_local["touching_edges"]))
print("用于最终回答的文档：")
display(pd.DataFrame([dict(title=docs[d]["title"], doc_id=d, similarity=round(s,3))
                     for d,s in example_local["doc_scores"]]))
print("引用的社区报告：",example_local["report_ids"])
print("给 Llama 的 Local 检索上下文预览：\n")
print(example_local["context"][:3200])
print("\nLocal 检索子图：")
selected_subgraph=G.subgraph(example_local["related_nodes"]).copy()
draw_knowledge_graph(selected_subgraph,title="Local Search · seeds and one-hop neighbors",limit=75)

Local 的种子实体： ['Robert Oliver Reed', 'Oliver Reed', 'Oliver!', 'Malcolm McDowell', 'Alan Bates', 'Richard Lester']
一跳扩展后实体数量： 24
涉及种子的关系数量： 27
用于最终回答的文档：


,title,doc_id,similarity
0,Royal Flash (film),1377ef4dedadd8193f4e,0.640
1,Oliver Reed,bb9945cb5ab48df85f2a,0.631
2,Royal Flash,ef3abbe53541a7e85146,0.478
3,Harry Flashman,4f44e2bddef4decbf9f4,0.456
4,The Duke of Hamilton,f31d370690ecda663912,0.386
5,Ivan Dragomiloff,eab198086fce65d88781,0.375
6,Lion of the Desert,c439b304cae6aa8cbb0c,0.351


引用的社区报告： [7, 20]
给 Llama 的 Local 检索上下文预览：

SELECTED GRAPH ENTITIES:

- Robert Oliver Reed: An English actor known for his macho image and tough guy roles.

- Oliver Reed: Actor photographed by Robin Barton.; Actor who played Ivan Dragomiloff in the film adaptation.; Actor who appeared as Otto von Bismarck in Royal Flash.; An actor known for frequent visits to The Duke of Hamilton.; Actor who portrayed Italian General Rodolfo Graziani in the film.

- Oliver!: A notable film released in 1968 featuring Oliver Reed.

- Malcolm McDowell: Actor who played Harry Flashman in the 1975 film 'Royal Flash'.; Actor who stars as Flashman in the film Royal Flash.

- Alan Bates: Actor who played Rudi von Sternberg in Royal Flash.

- Richard Lester: Director of the 1975 film 'Royal Flash' in which Harry Flashman was portrayed.; Director of the film Royal Flash.

RELATIONS TOUCHING SEED ENTITIES:

- Robin Barton <-> Oliver Reed: Robin Barton photographed Oliver Reed.

- Oliver Reed <-> Ivan Dragomiloff:

## 第 10 步｜手写 Global Search：社区报告选择 → GPT Map

先对问题与每份社区报告做 Embedding 相似度排序，选择最高的 `GLOBAL_TOP_COMMUNITIES` 份报告；然后**逐社区让 GPT-4o-mini 产出局部答案片段**。这里也先展示中间结果，暂时不加载 Llama。

注意：**Map 阶段已经产生候选答案**，因此这是 GraphRAG 的混合模型变体；最终 Reduce 才使用本地 Llama 70B。

In [34]:
MAP_SYS = """You are one MAP worker in a GraphRAG query-focused summarization system.
Use ONLY the supplied community report to answer the question as far as supported.
Return exactly one JSON object: {"relevance":0.0,"partial_answer":"brief evidence-based answer or IRRELEVANT", "support":["quote-free factual evidence summary"]}.
Use relevance between 0 and 1. Do not fabricate missing hops or use prior knowledge."""

def map_one(question,cid):
    cache_key=sha(json.dumps({"question":question,"cid":cid,"model":MAP_MODEL,
                              "report":report_text(reports[cid]),"max_tokens":MAX_MAP_OUTPUT},sort_keys=True))
    path=ROOT/"global_maps"/f"{cache_key}.json"
    if path.exists():return json.loads(path.read_text(encoding="utf-8"))
    for attempt in range(3):
        try:
            response=client.chat.completions.create(model=MAP_MODEL,temperature=0,
                max_tokens=MAX_MAP_OUTPUT,response_format={"type":"json_object"},
                messages=[{"role":"system","content":MAP_SYS},
                          {"role":"user","content":f"Question: {question}\n\nCommunity {cid}:\n{report_text(reports[cid])}"}])
            obj=json.loads(response.choices[0].message.content)
            result=dict(community_id=cid, relevance=float(obj.get("relevance",0)),
                partial_answer=str(obj.get("partial_answer", "IRRELEVANT")),
                support=[str(s) for s in (obj.get("support") or [])][:8],
                usage=usage_dict(response.usage))
            atomic_json(path,result)
            return result
        except Exception:
            if attempt==2:raise
            time.sleep(2**attempt)

def global_map(question):
    qvec=embed_question(question)
    ranking=rank_vectors(qvec,report_vectors,report_ids,GLOBAL_TOP_COMMUNITIES)
    selected=[cid for cid,_ in ranking]
    found={}
    with ThreadPoolExecutor(max_workers=MAP_WORKERS) as pool:
        work={pool.submit(map_one,question,cid):cid for cid in selected}
        for future in tqdm(as_completed(work),total=len(work),desc="GPT Global Map"):
            found[work[future]]=future.result()
    maps=[found[cid] for cid in selected]  # 保持向量排名顺序
    useful=[r for r in maps if r["partial_answer"].strip()
            and r["partial_answer"].strip().upper()!="IRRELEVANT" and r["relevance"]>0]
    fragments=[f"[Community {r['community_id']} | relevance={r['relevance']:.2f}] {r['partial_answer']}"
               for r in useful]
    context="\n\n".join(fragments) if fragments else "No community reports yielded relevant partial answers."
    source_ids=sorted({d for x in useful for d in communities[x["community_id"]]["doc_ids"]})
    return dict(report_ranking=ranking,maps=maps,useful=useful,
        doc_ids=source_ids, context=context)

example_global=global_map(q_example)
print("选择的报告及相似度：")
display(pd.DataFrame([dict(community_id=cid,title=reports[cid]["title"],similarity=round(s,3))
                     for cid,s in example_global["report_ranking"]]))
print("Map 阶段生成的部分答案（每个社区一行）：")
display(pd.DataFrame([dict(community_id=r["community_id"],relevance=r["relevance"],
    partial_answer=r["partial_answer"][:450],
    supports=" | ".join(r["support"])[:350]) for r in example_global["maps"]]))
print("\n送给本地 Llama Reduce 的输入预览：\n")
print(example_global["context"][:3200])

GPT Global Map:   0%|          | 0/8 [00:00<?, ?it/s]

选择的报告及相似度：


,community_id,title,similarity
0,20,Community 20: Oliver Reed and Funny Bones,0.496
1,7,Community 7,0.407
2,42,Community 42,0.277
3,32,Alfred Hurley Community,0.272
4,29,Community 29: John Braddocke and His Connections,0.256
5,1,Community Overview of Twisted Fortune and Rela...,0.237
6,39,Community 39,0.229
7,37,Operation Quicksilver and Deception in WWII,0.227


Map 阶段生成的部分答案（每个社区一行）：


,community_id,relevance,partial_answer,supports
0,20,0.0,IRRELEVANT,The community report does not provide informat...
1,7,0.0,IRRELEVANT,The community report does not provide informat...
2,42,0.0,IRRELEVANT,The community report does not provide any info...
3,32,0.0,IRRELEVANT,The community report does not provide any info...
4,29,0.0,IRRELEVANT,The community report does not provide any info...
5,1,0.0,IRRELEVANT,The community report does not provide any info...
6,39,0.0,IRRELEVANT,The community report does not provide any info...
7,37,0.0,IRRELEVANT,The community report does not provide any info...



送给本地 Llama Reduce 的输入预览：

No community reports yielded relevant partial answers.


## 第 11 步｜**现在才加载本地 Llama 3.3 70B**

此前的建图、社区报告、向量和 Global Map 已经完成，不需要在这些阶段占据 A100 显存。这个阶段只让 70B 读取检索结果，生成最终简短答案。

Colab 选择 **A100 80 GB GPU**。首次下载 4-bit 模型可能需要一段时间；如果 Hugging Face 模型访问受限，请确认账号已获得模型使用许可。不要把 Hugging Face Token 填到 OpenAI Key 提示里。

In [36]:
import sys, subprocess, importlib.util
requirements = ["transformers>=4.51,<5", "accelerate>=1.7,<2", "bitsandbytes>=0.46,<1"]
missing = [x for x in requirements if importlib.util.find_spec(x.split(">=")[0]) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *requirements])

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 GPU，请在 Colab 选择 A100 运行时后重试")
print("GPU:",torch.cuda.get_device_name(0))
try:
    from google.colab import userdata
    hf_token=userdata.get("HF_TOKEN")
except Exception:
    hf_token=None
hf_token=(hf_token or os.getenv("HF_TOKEN") or "").strip()
if not hf_token:
    hf_token=getpass("Hugging Face Token (hf_...): ").strip()
if not hf_token.startswith("hf_"):
    raise ValueError("这里需要 hf_ 开头的 Hugging Face Token")

print("加载本地 4-bit Llama 3.3 70B（首次下载可能较慢）...")
llama_tokenizer=AutoTokenizer.from_pretrained(FINAL_LLM, token=hf_token)
llama_model = AutoModelForCausalLM.from_pretrained(
    FINAL_LLM,
    token=hf_token,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)
llama_model.eval()
if llama_tokenizer.pad_token_id is None:
    llama_tokenizer.pad_token_id=llama_tokenizer.eos_token_id
stop_ids=[llama_tokenizer.eos_token_id]
eot=llama_tokenizer.convert_tokens_to_ids("<|eot_id|>")
if isinstance(eot,int) and eot not in stop_ids and eot != llama_tokenizer.unk_token_id:
    stop_ids.append(eot)
print("Llama 已加载；占用显存：",round(torch.cuda.memory_allocated()/2**30,2),"GiB")

def llama_answer(system_prompt,user_prompt):
    messages=[{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}]
    tok=llama_tokenizer.apply_chat_template(messages,add_generation_prompt=True,return_tensors="pt")
    if not torch.is_tensor(tok):tok=tok["input_ids"]
    in_len=tok.shape[-1]
    if in_len>MAX_LLAMA_INPUT:
        raise ValueError(f"最终输入 {in_len} tokens 超过 {MAX_LLAMA_INPUT}；请缩小 LOCAL_MAX_DOCS / 报告数量")
    tok=tok.to(next(llama_model.parameters()).device)
    with torch.inference_mode():
        output=llama_model.generate(tok,do_sample=False,max_new_tokens=MAX_LLAMA_OUTPUT,
            eos_token_id=stop_ids,pad_token_id=llama_tokenizer.pad_token_id)
    cont=output[0][in_len:]
    answer=llama_tokenizer.decode(cont,skip_special_tokens=True).strip()
    return answer, dict(prompt_tokens=int(in_len),completion_tokens=int(cont.numel()),
                        total_tokens=int(in_len+cont.numel()))

LOCAL_FINAL_SYS=("Answer multi-hop questions strictly based on retrieved evidence. "
    "Return only the short final answer, no explanation. "
    "If insufficient evidence, respond 'Cannot be determined from the provided context'.")
GLOBAL_FINAL_SYS=("Synthesize the final short answer to the question from community partial answers. "
    "Return only the answer, no explanation. Do not invent unsupported facts.")
print("已准备好 Local 最终生成 / Global Reduce")

GPU: NVIDIA A100-SXM4-80GB
Hugging Face Token (hf_...): ··········
加载本地 4-bit Llama 3.3 70B（首次下载可能较慢）...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Llama 已加载；占用显存： 69.7 GiB
已准备好 Local 最终生成 / Global Reduce


## 第 12 步｜比较同一道 HotpotQA 问题的两种回答

到这里终于调用 70B：**Local** 的输入是图邻居、社区和原文；**Global** 的输入是 GPT Map 得到的社区局部答案。显示最终回答、EM/F1 和不同模型的 query-time token。

In [37]:
def normalize_answer(value):
    text=str(value).lower()
    text="".join(c for c in text if c not in string.punctuation)
    text=re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def exact_match(p,g):return int(normalize_answer(p)==normalize_answer(g))

def answer_f1(pred,gold):
    p,g=normalize_answer(pred),normalize_answer(gold)
    specials={"yes","no","noanswer"}
    if (p in specials or g in specials) and p!=g:return 0.0
    pc,gc=Counter(p.split()),Counter(g.split())
    common=sum((pc & gc).values())
    if not common:return 0.0
    precision=common/max(1,sum(pc.values()))
    recall=common/max(1,sum(gc.values()))
    return 2*precision*recall/(precision+recall)

assert exact_match("The USA", "usa") == 1
assert answer_f1("yes","no")==0

def run_final(row,method, retrieval):
    if method=="local":
        prompt=f"Question: {row['question']}\n\nEVIDENCE:\n{retrieval['context']}"
        model_sys=LOCAL_FINAL_SYS
        map_usage=dict(prompt_tokens=0,completion_tokens=0,total_tokens=0)
    elif method=="global":
        prompt=f"Question: {row['question']}\n\nCOMMUNITY MAP RESULTS:\n{retrieval['context']}"
        model_sys=GLOBAL_FINAL_SYS
        map_usage={k:sum(r['usage'][k] for r in retrieval['maps'])
                   for k in ("prompt_tokens","completion_tokens","total_tokens")}
    else:raise ValueError(method)
    text, llama_usage=llama_answer(model_sys,prompt)
    if not text: raise ValueError("Llama returned empty answer")
    evidence=set(retrieval["doc_ids"])
    gold=set(gold_support_docs[row["id"]])
    return dict(id=row["id"],method=method,question=row["question"],
        gold_answer=row["answer"],prediction=text,EM=exact_match(text,row["answer"]),
        F1=answer_f1(text,row["answer"]),
        document_evidence_recall=len(evidence&gold)/len(gold) if gold else np.nan,
        all_support_docs_retrieved=int(gold.issubset(evidence)) if gold else np.nan,
        retrieved_doc_ids=json.dumps(retrieval["doc_ids"]),
        llama_prompt_tokens=llama_usage["prompt_tokens"],
        llama_completion_tokens=llama_usage["completion_tokens"],
        llama_total_tokens=llama_usage["total_tokens"],
        gpt_map_prompt_tokens=map_usage["prompt_tokens"],
        gpt_map_completion_tokens=map_usage["completion_tokens"],
        gpt_map_total_tokens=map_usage["total_tokens"])

sample_comparison=[]
for method,retrieval in [("local",example_local),("global",example_global)]:
    started=time.perf_counter()
    answer=run_final(questions[0],method,retrieval)
    answer["latency_sec"]=time.perf_counter()-started
    sample_comparison.append(answer)

print("问题：",questions[0]["question"])
print("Gold：",questions[0]["answer"])
display(pd.DataFrame(sample_comparison)[[
    "method","prediction","EM","F1","document_evidence_recall",
    "llama_total_tokens","gpt_map_total_tokens","latency_sec"]])

问题： What nationality was Oliver Reed's character in the film Royal Flash?
Gold： Prussian


,method,prediction,EM,F1,document_evidence_recall,llama_total_tokens,gpt_map_total_tokens,latency_sec
0,local,German,0,0.0,0.5,2007,0,1.715776
1,global,British,0,0.0,0.0,93,2965,0.449347


## 第 13 步｜可选：在 N 道题上完整评测 + 断点续跑

默认 **不自动跑完整评估**，因为你现在主要在学习每个阶段。要评估当前的 N 道题，将 `RUN_EVALUATION=True`。每题每方法成功后写入 CSV，重跑自动跳过；失败会记录 error 并允许下次重试。正式 200 条时先把第 1 步 N 改为 200，重新构建相应索引。

In [40]:
RUN_EVALUATION = True  # 想批量评估时改为 True

def evaluate_method(method):
    path=ROOT/"predictions"/f"{method}_n{N_QUESTIONS}.csv"
    previous=pd.read_csv(path,dtype={"id":str}) if path.exists() else pd.DataFrame()
    stored={str(r["id"]):r for r in previous.to_dict("records")} if not previous.empty else {}
    for q in tqdm(questions,desc="Eval " + method):
        old=stored.get(q["id"])
        if old and str(old.get("error","")) in ("","nan") and str(old.get("prediction","")) not in ("","nan"):
            continue
        t0=time.perf_counter()
        try:
            retrieval=local_retrieve(q["question"]) if method=="local" else global_map(q["question"])
            result=run_final(q,method,retrieval)
            result["error"]=""
        except Exception as exc:
            print("Question failed, will retry next time:",q["id"],repr(exc))
            result=dict(id=q["id"],method=method,question=q["question"],
                gold_answer=q["answer"],prediction="",EM=0,F1=0,error=repr(exc))
        result["latency_sec"]=round(time.perf_counter()-t0,3)
        stored[q["id"]]=result
        rows=[stored[x["id"]] for x in questions if x["id"] in stored]
        temp=path.with_suffix(".tmp")
        pd.DataFrame(rows).to_csv(temp,index=False)
        os.replace(temp,path)
    return pd.DataFrame([stored[x["id"]] for x in questions if x["id"] in stored])

if RUN_EVALUATION:
    eval_results={method:evaluate_method(method) for method in ("local","global")}
    summary=[]
    for method,df in eval_results.items():
        good=df["error"].fillna("").eq("") & df["prediction"].fillna("").astype(str).str.len().gt(0)
        numeric=lambda key:pd.to_numeric(df.get(key,pd.Series(np.nan,index=df.index)),errors="coerce")
        summary.append(dict(method=method,n=len(df),successful=int(good.sum()),
            EM=numeric("EM").mean(),F1=numeric("F1").mean(),
            document_evidence_recall=numeric("document_evidence_recall")[good].mean(),
            avg_llama_tokens=numeric("llama_total_tokens")[good].mean(),
            avg_gpt_map_tokens=numeric("gpt_map_total_tokens")[good].mean(),
            avg_seconds=numeric("latency_sec")[good].mean()))
    final_summary=pd.DataFrame(summary)
    display(final_summary.round(4))
    final_summary.to_csv(ROOT/"predictions"/"summary.csv",index=False)
else:
    print("仅学习中间步骤。本单元格未运行批量评估。改 RUN_EVALUATION=True 可开始。")

Eval local:   0%|          | 0/20 [00:00<?, ?it/s]

Embedding question_563b259d50ef060c9728:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding question_400ff45cbe4d08cddca2:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding question_6558000e1c9f37bda1b3:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding question_54d24d24bc62f4b187b1:   0%|          | 0/1 [00:00<?, ?it/s]

...

GPT Global Map:   0%|          | 0/8 [00:00<?, ?it/s]

GPT Global Map:   0%|          | 0/8 [00:00<?, ?it/s]

GPT Global Map:   0%|          | 0/8 [00:00<?, ?it/s]

GPT Global Map:   0%|          | 0/8 [00:00<?, ?it/s]

,method,n,successful,EM,F1,document_evidence_recall,avg_llama_tokens,avg_gpt_map_tokens,avg_seconds
0,local,20,20,0.6,0.7465,0.90,1870.25,0.0,2.3479
1,global,20,20,0.3,0.4598,0.35,118.25,2858.7,3.1465


## 第 14 步｜结果、代价、可重复性与方法局限

**打开 Google Drive 中本次的 `ROOT`：**

- `corpus.json`：原始候选文档与题目映射（没有 gold）。
- `extractions/<doc_id>.json`：每篇文档的**真实实体、关系、LLM usage、原始 JSON**。
- `graph_index.json`、`graph_edges.graphml`：合并后的图、社区编号与证据映射。
- `figures/entity_communities.html`：可交互的知识图谱。
- `reports/community_<id>.json`：社区报告标题、正文和 key findings。
- `embeddings/*.npz`：向量缓存。
- `global_maps/*.json`：各题每社区 Map 的答案片段。
- `predictions/*.csv`：逐题 Local / Global / EM / F1（仅批量评估开启后生成）。

**阅读结果时注意：**

1. 这里是**手写教学版**，社区划分为单层 Leiden，并非原论文的层次聚类；Global 仅对 Top-K 社区做 Map。
2. 数据集使用的是题目提供的候选文档构成的**共享语料图**。它与单题上下文严格隔离的 HotpotQA 阅读理解条件不同。
3. `document_evidence_recall` 是**文档级**，不是 HotpotQA 官方的句子级 Supporting Facts F1；Global 社区来源集合也不能视为逐句准确归因。
4. Map 过程已经使用 GPT 生成局部答案，因此固定的是**最终生成模型**而非总推理预算。OpenAI Embedding 与构图/报告也是额外计算成本。
5. 本 Notebook 中**真实 Llama / GPT 调用需在你的 Colab 运行**；不能把代码语法或模拟测试当成真实实验成绩。

In [39]:
# 可观察不同阶段累计的真实 OpenAI 使用量（命中缓存的调用不再计费）
extraction_total=sum(int(x.get("usage",{}).get("total_tokens",0)) for x in extractions.values())
report_total=sum(int(x.get("usage",{}).get("total_tokens",0)) for x in reports.values())
print("GPT extraction recorded tokens:",extraction_total)
print("GPT community report recorded tokens:",report_total)
print("Embedding 本次新增 tokens:",sum(x.get("total_tokens_new",0) for x in embedding_usage.values()))
print("注：前两项是缓存产物中保存的总量；Embedding '本次新增' 不代表历史完整用量。")
print("请在 OpenAI 后台查看真实账单；这里不估计未经验证的 API 价格。")
print("\n本次全部产物保存在：",ROOT)

GPT extraction recorded tokens: 193183
GPT community report recorded tokens: 93130
Embedding 本次新增 tokens: 72040
注：前两项是缓存产物中保存的总量；Embedding '本次新增' 不代表历史完整用量。
请在 OpenAI 后台查看真实账单；这里不估计未经验证的 API 价格。

本次全部产物保存在： /content/drive/MyDrive/GraphRAG_HotpotQA_Tutorial/hotpot_n20_seed42_9c3a52009c3d
